<a href="https://colab.research.google.com/github/hgshreyas/BIS_LAB/blob/main/1BM24CS108_Drone_Delivery_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import random
from collections import deque

# ---------------------------------------
# 1. USER INPUT
# ---------------------------------------

ROWS = int(input("Enter number of rows: "))
COLS = int(input("Enter number of columns: "))

print("Enter starting position (row column):")
START = tuple(map(int, input().split()))

print("Enter destination position (row column):")
GOAL = tuple(map(int, input().split()))

grid = [[0 for j in range(COLS)] for i in range(ROWS)]

# Enter buildings
b = int(input("Enter number of buildings: "))

print("Enter building positions (row column):")

for i in range(b):
    x, y = map(int, input().split())
    grid[x][y] = 1

# Enter restricted areas
r = int(input("Enter number of restricted areas: "))

print("Enter restricted area positions (row column):")

for i in range(r):
    x, y = map(int, input().split())
    grid[x][y] = 2

# PCA parameters
CELL_ROWS = int(input("Enter number of cell rows: "))
CELL_COLS = int(input("Enter number of cell columns: "))
ITERATIONS = int(input("Enter number of iterations: "))

# Validate input
if not (0 <= START[0] < ROWS and 0 <= START[1] < COLS):
    raise ValueError("Invalid starting position")

if not (0 <= GOAL[0] < ROWS and 0 <= GOAL[1] < COLS):
    raise ValueError("Invalid destination")

if grid[START[0]][START[1]] != 0 or grid[GOAL[0]][GOAL[1]] != 0:
    raise ValueError("Start and destination must be free cells")

if CELL_ROWS < 1 or CELL_COLS < 1 or ITERATIONS < 1:
    raise ValueError("PCA parameters must be positive")


# ---------------------------------------
# 2. FIND VALID NEIGHBOURS
# ---------------------------------------

def get_neighbors(cell):

    x, y = cell

    moves = [
        (x + 1, y),
        (x - 1, y),
        (x, y + 1),
        (x, y - 1)
    ]

    valid = []

    for a, b in moves:

        if (0 <= a < ROWS and
            0 <= b < COLS and
            grid[a][b] == 0):

            valid.append((a, b))

    random.shuffle(valid)

    return valid


# ---------------------------------------
# 3. GENERATE VALID PATH USING BFS
# ---------------------------------------

def find_path(start, goal):

    queue = deque([[start]])
    visited = {start}

    while queue:

        path = queue.popleft()
        current = path[-1]

        if current == goal:
            return path

        for neighbor in get_neighbors(current):

            if neighbor not in visited:

                visited.add(neighbor)
                queue.append(path + [neighbor])

    return None


# ---------------------------------------
# 4. FITNESS FUNCTION
# ---------------------------------------

def fitness(path):

    if path is None:
        return float('inf')

    distance = len(path) - 1
    turns = 0

    for i in range(1, len(path) - 1):

        x1, y1 = path[i - 1]
        x2, y2 = path[i]
        x3, y3 = path[i + 1]

        if ((x2-x1, y2-y1) !=
            (x3-x2, y3-y2)):

            turns += 1

    energy = distance + 0.5 * turns

    cost = distance + 0.5 * energy

    return cost


# ---------------------------------------
# 5. IMPROVE PATH
# ---------------------------------------

def improve_path(path):

    if len(path) < 4:
        return path[:]

    i = random.randint(0, len(path) - 3)
    j = random.randint(i + 2, len(path) - 1)

    new_segment = find_path(path[i], path[j])

    if new_segment is None:
        return path[:]

    new_path = path[:i] + new_segment + path[j+1:]

    if fitness(new_path) < fitness(path):
        return new_path

    return path[:]


# ---------------------------------------
# 6. INITIALIZE CELL POPULATION
# ---------------------------------------

population = []

for i in range(CELL_ROWS):

    row = []

    for j in range(CELL_COLS):

        path = find_path(START, GOAL)
        row.append(path)

    population.append(row)


# Check whether a valid path exists
if any(path is None for row in population for path in row):
    raise ValueError("No valid path exists between start and destination")


best_path = min(
    [path for row in population for path in row],
    key=fitness
)


# ---------------------------------------
# 7. PARALLEL CELLULAR ALGORITHM
# ---------------------------------------

for iteration in range(ITERATIONS):

    old_population = [
        [path[:] for path in row]
        for row in population
    ]

    new_population = []

    for i in range(CELL_ROWS):

        new_row = []

        for j in range(CELL_COLS):

            neighbors = []

            # Moore neighbourhood
            for a in range(max(0, i-1),
                           min(CELL_ROWS, i+2)):

                for b in range(max(0, j-1),
                               min(CELL_COLS, j+2)):

                    neighbors.append(old_population[a][b])

            best_neighbor = min(neighbors, key=fitness)

            candidate = improve_path(best_neighbor)

            choices = [
                old_population[i][j],
                best_neighbor,
                candidate
            ]

            selected = min(choices, key=fitness)

            new_row.append(selected[:])

        new_population.append(new_row)

    population = new_population

    for row in population:

        for path in row:

            if fitness(path) < fitness(best_path):
                best_path = path[:]


# ---------------------------------------
# 8. DISPLAY RESULTS
# ---------------------------------------

distance = len(best_path) - 1
turns = 0

for i in range(1, len(best_path)-1):

    x1, y1 = best_path[i-1]
    x2, y2 = best_path[i]
    x3, y3 = best_path[i+1]

    if ((x2-x1, y2-y1) !=
        (x3-x2, y3-y2)):

        turns += 1

energy = distance + 0.5 * turns

print("\n========== PCA DRONE DELIVERY ==========")

print("\nOptimized Drone Path:")

for cell in best_path:
    print(cell, end=" -> ")

print("END")

print("\nDistance:", distance)
print("Number of Turns:", turns)
print("Energy Consumption:", round(energy, 2))
print("Minimum Fitness Cost:", round(fitness(best_path), 2))


# ---------------------------------------
# 9. DISPLAY GRID
# ---------------------------------------

print("\n========== DRONE DELIVERY GRID ==========")

for i in range(ROWS):

    for j in range(COLS):

        cell = (i, j)

        if cell == START:
            print("S", end=" ")

        elif cell == GOAL:
            print("G", end=" ")

        elif cell in best_path:
            print("*", end=" ")

        elif grid[i][j] == 1:
            print("X", end=" ")

        elif grid[i][j] == 2:
            print("R", end=" ")

        else:
            print(".", end=" ")

    print()

print("\nS = Start")
print("G = Destination")
print("* = Drone Path")
print("X = Building")
print("R = Restricted Area")
print(". = Free Cell")


Enter number of rows: 8
Enter number of columns: 8
Enter starting position (row column):
0 0 
Enter destination position (row column):
7 7
Enter number of buildings: 8
Enter building positions (row column):
1 1
1 2
1 3
2 3
3 3
4 3
5 3
6 5
Enter number of restricted areas: 2
Enter restricted area positions (row column):
5 5
3 6
Enter number of cell rows: 4
Enter number of cell columns: 4
Enter number of iterations: 30

========== PCA DRONE DELIVERY ==========

Optimized Drone Path:
(0, 0) -> (1, 0) -> (2, 0) -> (3, 0) -> (4, 0) -> (5, 0) -> (6, 0) -> (7, 0) -> (7, 1) -> (7, 2) -> (7, 3) -> (7, 4) -> (7, 5) -> (7, 6) -> (7, 7) -> END

Distance: 14
Number of Turns: 1
Energy Consumption: 14.5
Minimum Fitness Cost: 21.25

========== DRONE DELIVERY GRID ==========
S . . . . . . . 
* X X X . . . . 
* . . X . . . . 
* . . X . . R . 
* . . X . . . . 
* . . X . R . . 
* . . . . X . . 
* * * * * * * G 

S = Start
G = Destination
* = Drone Path
X = Building
R = Restricted Area
. = Free Cell
